In [4]:

# — Connect to Database

import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Connect to the database you created in Phase 1
conn = sqlite3.connect('../data/fintech_churn.db')
print("✅ Connected to fintech_churn.db")

# Quick sanity check — confirm all 4 tables exist with correct row counts
print("\n📋 Table sizes:")
for table in ['users', 'transactions', 'app_events', 'support_tickets']:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    print(f"   {table:<25} {count:>8,} rows")

✅ Connected to fintech_churn.db

📋 Table sizes:
   users                       10,000 rows
   transactions               144,355 rows
   app_events                 228,416 rows
   support_tickets              3,807 rows


In [5]:
cohort_query = """
WITH user_cohorts AS (
    SELECT
        user_id,
        signup_date,
        STRFTIME('%Y-%m', signup_date) AS cohort_month
    FROM users
),
user_activity AS (
    SELECT
        t.user_id,
        t.txn_date
    FROM transactions t
    WHERE t.status = 'success'
),
cohort_activity AS (
    SELECT
        c.user_id,
        c.cohort_month,
        CAST(
            (CAST(STRFTIME('%Y', a.txn_date) AS INTEGER) - CAST(STRFTIME('%Y', c.signup_date) AS INTEGER)) * 12 +
            (CAST(STRFTIME('%m', a.txn_date) AS INTEGER) - CAST(STRFTIME('%m', c.signup_date) AS INTEGER))
        AS INTEGER) AS months_since_signup
    FROM user_cohorts c
    JOIN user_activity a ON c.user_id = a.user_id
)
SELECT cohort_month, months_since_signup, COUNT(DISTINCT user_id) AS retained_users
FROM cohort_activity
WHERE months_since_signup BETWEEN 0 AND 6
GROUP BY cohort_month, months_since_signup
ORDER BY cohort_month, months_since_signup
"""

df_cohort = pd.read_sql(cohort_query, conn)
print("Sample rows:")
print(df_cohort.head(10))
print("\nUnique months_since_signup values:", sorted(df_cohort['months_since_signup'].unique()))

# Get total users per cohort (this is our denominator — the TRUE 100%)
cohort_sizes = pd.read_sql("""
    SELECT STRFTIME('%Y-%m', signup_date) AS cohort_month, COUNT(*) AS cohort_size
    FROM users
    GROUP BY cohort_month
""", conn).set_index('cohort_month')['cohort_size']

cohort_pivot = df_cohort.pivot_table(
    index='cohort_month', columns='months_since_signup',
    values='retained_users', fill_value=0
)

print("\nPivot columns found:", cohort_pivot.columns.tolist())

cohort_pct = cohort_pivot.div(cohort_sizes, axis=0).round(3) * 100
cohort_pct.columns = [f'Month {c}' for c in cohort_pct.columns]

print("\n📊 Cohort Retention Matrix (% of cohort's total users active in month N):")
print(cohort_pct.to_string())

Sample rows:
  cohort_month  months_since_signup  retained_users
0      2024-12                    6               9
1      2025-01                    5              23
2      2025-01                    6             151
3      2025-02                    4              25
4      2025-02                    5             112
5      2025-02                    6             204
6      2025-03                    3              13
7      2025-03                    4             125
8      2025-03                    5             218
9      2025-03                    6             289

Unique months_since_signup values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Pivot columns found: [0, 1, 2, 3, 4, 5, 6]

📊 Cohort Retention Matrix (% of cohort's total users active in month N):
              Month 0  Month 1  Month 2  Month 3  Month 4  Month 5  Month 6
cohort_month                                                               
2024-12          

In [6]:

# — RFM Segmentation

rfm_query = """
WITH rfm_base AS (
    SELECT
        u.user_id,
        u.churn,
        -- Recency: days since last successful transaction
        JULIANDAY('now') - JULIANDAY(MAX(t.txn_date)) AS recency_days,
        -- Frequency: number of successful transactions
        COUNT(t.txn_id)                                AS frequency,
        -- Monetary: total amount spent
        ROUND(SUM(t.amount), 2)                        AS monetary
    FROM users u
    LEFT JOIN transactions t
           ON u.user_id = t.user_id AND t.status = 'success'
    GROUP BY u.user_id, u.churn
)

SELECT
    user_id,
    churn,
    ROUND(recency_days, 0)  AS recency_days,
    frequency,
    COALESCE(monetary, 0)   AS monetary,

    -- Score each metric 1–4 using CASE (1 = worst, 4 = best)
    CASE
        WHEN recency_days <= 30  THEN 4
        WHEN recency_days <= 90  THEN 3
        WHEN recency_days <= 180 THEN 2
        ELSE 1
    END AS r_score,

    CASE
        WHEN frequency >= 20 THEN 4
        WHEN frequency >= 10 THEN 3
        WHEN frequency >= 3  THEN 2
        ELSE 1
    END AS f_score,

    CASE
        WHEN monetary >= 50000 THEN 4
        WHEN monetary >= 10000 THEN 3
        WHEN monetary >= 1000  THEN 2
        ELSE 1
    END AS m_score

FROM rfm_base
"""

df_rfm = pd.read_sql(rfm_query, conn)

# ── Combine scores into RFM segment labels ──
def rfm_segment(row):
    score = row['r_score'] + row['f_score'] + row['m_score']
    if score >= 11:
        return 'Champion'
    elif score >= 9:
        return 'Loyal'
    elif score >= 7:
        return 'Promising'
    elif score >= 5:
        return 'At Risk'
    else:
        return 'Lost'

df_rfm['segment'] = df_rfm.apply(rfm_segment, axis=1)

# ── Summary: segment distribution ──
segment_summary = df_rfm.groupby(['segment', 'churn']).size().unstack(fill_value=0)
segment_summary.columns = ['Active', 'Churned']
segment_summary['Total']       = segment_summary['Active'] + segment_summary['Churned']
segment_summary['Churn Rate %'] = (segment_summary['Churned'] / segment_summary['Total'] * 100).round(1)

print("📊 RFM Segment Distribution:")
print(segment_summary.sort_values('Churn Rate %', ascending=False).to_string())



# CELL 3B — Funnel Drop-off Analysis
# Question: At which step do users abandon the transaction flow?


funnel_query = """
SELECT
    -- Step 1: Users who opened the app
    COUNT(DISTINCT CASE WHEN event_type = 'app_open'       THEN user_id END) AS s1_app_open,
    -- Step 2: Users who reached home screen
    COUNT(DISTINCT CASE WHEN event_type = 'home_screen'    THEN user_id END) AS s2_home_screen,
    -- Step 3: Users who initiated a transaction
    COUNT(DISTINCT CASE WHEN event_type = 'txn_initiated'  THEN user_id END) AS s3_txn_initiated,
    -- Step 4: Users who completed a transaction
    COUNT(DISTINCT CASE WHEN event_type = 'txn_success'    THEN user_id END) AS s4_txn_success
FROM app_events
"""

df_funnel = pd.read_sql(funnel_query, conn)

# ── Calculate drop-off at each step ──
steps  = ['s1_app_open', 's2_home_screen', 's3_txn_initiated', 's4_txn_success']
labels = ['App Open', 'Home Screen', 'Txn Initiated', 'Txn Success']
values = [df_funnel[s].iloc[0] for s in steps]

print("\n📊 Funnel Drop-off Analysis:")
print(f"   {'Step':<20} {'Users':>8}   {'Drop-off':>10}")
print("   " + "-" * 42)
for i, (label, val) in enumerate(zip(labels, values)):
    if i == 0:
        dropoff = "—  (baseline)"
    else:
        lost = values[i-1] - val
        pct  = (lost / values[i-1]) * 100
        dropoff = f"-{lost:,} users  ({pct:.1f}% lost)"
    print(f"   {label:<20} {val:>8,}   {dropoff}")

conn.close()
print("\n✅ Phase 2 SQL Analysis complete!")

📊 RFM Segment Distribution:
           Active  Churned  Total  Churn Rate %
segment                                        
Lost            0        9      9         100.0
At Risk        30      247    277          89.2
Promising    1036      531   1567          33.9
Loyal        7571       41   7612           0.5
Champion      535        0    535           0.0

📊 Funnel Drop-off Analysis:
   Step                    Users     Drop-off
   ------------------------------------------
   App Open               10,000   —  (baseline)
   Home Screen             9,989   -11 users  (0.1% lost)
   Txn Initiated           9,707   -282 users  (2.8% lost)
   Txn Success             9,420   -287 users  (3.0% lost)

✅ Phase 2 SQL Analysis complete!
